# 32 — Fairness & Bias in AI — Lab

## Description

This lab audits a lending classifier for fairness disparities. You will load the Adult Income dataset, train a logistic regression classifier, compute group fairness metrics using the AIF360 library, identify the likely bias taxonomy category explaining the observed disparity, and write a brief audit report. The hands-on work reinforces how bias enters ML pipelines and how fairness metrics quantify disparity.

## Objectives

- Classify a given bias source into its category (historical, representation, measurement, aggregation) and trace it to the specific ML pipeline stage where it enters.
- Compute Demographic Parity difference, Equalized Odds difference, and disparate impact ratio for a classifier using AIF360.

## Prerequisites

You should have completed the **Lecture 32** and be familiar with these concepts: bias taxonomy (historical, representation, measurement, aggregation), group fairness criteria (Demographic Parity, Equalized Odds, Equal Opportunity), protected attributes, true positive rate (TPR), false positive rate (FPR), and basic supervised learning (training, testing, binary classification). Familiarity with **Lesson 03** (neural networks and loss functions) and **Lesson 07** (threat modeling) is assumed from the lesson plan's "builds-on" list.

## Required Software/Packages

- Python 3.8+
- pandas (data manipulation; use pandas < 3.0 — aif360 0.6.x is not yet compatible with pandas 3)
- numpy (numerical operations)
- scikit-learn (model training and evaluation)
- aif360 (fairness metrics and algorithms from IBM)

These are provided in the standard environment. If not, install aif360 via: `pip install aif360`.

## Environment/Setup Instructions

Run the import cell below to load the required libraries. This cell also sets a random seed for reproducibility.

**Dataset download**: `AdultDataset` requires the raw UCI Adult files. If the dataset cell fails with "No such file or directory", download [adult.data](https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data), [adult.test](https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.test), and [adult.names](https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.names) and place them, as-is, in the folder `aif360/data/raw/adult/` inside your aif360 installation (the error message prints the exact expected path).

In [ ]:
# Setup: Import libraries and set random seed for reproducibility
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from aif360.datasets import AdultDataset
from aif360.metrics import ClassificationMetric
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
print("Environment setup complete. All libraries imported.")

## Background/Context

Real-world ML pipelines often encode historical bias or structural inequities in their training data. The Adult Income dataset, used here, is derived from the 1994 US Census and contains demographic information and income labels (income > 50K or <= 50K). A lending classifier trained on historical lending decisions will likely reflect past discrimination. Your task is to measure and diagnose disparity using AIF360, a toolkit for fairness auditing. You will compute three fairness metrics, identify which bias taxonomy category best explains the disparity, and assess whether the model meets a fairness threshold.

## Exercise Instructions

1. Load the Adult Income dataset and confirm the protected attribute (gender) is available; print dataset summary statistics including gender distribution and positive-outcome rate by gender. (6 min)

2. Use AIF360 ClassificationMetric to compute: (a) Demographic Parity difference, (b) Equalized Odds difference, and (c) Disparate Impact ratio by gender; print all three metrics with their numeric values. (12 min)

3. Identify which bias taxonomy category (historical, representation, measurement, aggregation) most plausibly explains the observed disparity; cite at least one specific feature from the dataset that supports your categorization and explain the mechanism (e.g., "the feature AGE may serve as a proxy for another protected characteristic" or "historical hiring records embed past discrimination in the OCCUPATION column"). (6 min)

4. Write a three-sentence audit summary stating the numeric values of the three metrics, the identified bias taxonomy category, and a final verdict: whether the model meets a Demographic Parity difference threshold of <= 0.1 (yes or no). (6 min)

In [ ]:
# Starter Code: Load the Adult Income dataset and prepare data
# The AdultDataset from aif360 automatically loads and preprocesses the Adult dataset
adult_data = AdultDataset(
    label_name='income-per-year',
    favorable_classes=['>50K'],
    protected_attribute_names=['sex'],
    privileged_classes=[['Male']],
    categorical_features=['workclass', 'marital-status', 'occupation', 'relationship', 'race', 'native-country'],
    features_to_keep=['age', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week', 'sex', 'workclass', 'occupation', 'marital-status', 'race']
)

# Split into train and test
(dataset_train, dataset_test) = adult_data.split([0.7], shuffle=True)

print("Dataset Loaded Successfully")
print(f"Total samples: {len(adult_data.features)}")
print(f"Training samples: {len(dataset_train.features)}")
print(f"Test samples: {len(dataset_test.features)}")
print(f"\nDataset shape: {adult_data.features.shape}")
print(f"Protected attribute (sex): {adult_data.protected_attribute_names}")

In [ ]:
# Step 1: Confirm protected attribute and print summary statistics
# Extract the sex attribute and income labels from the test set
sex_index = adult_data.protected_attribute_names.index('sex')
sex_vals = dataset_test.protected_attributes[:, sex_index]
labels = dataset_test.labels.flatten()

# Create a dataframe for easy analysis
df_test = pd.DataFrame({
    'sex': sex_vals,
    'income': labels
})

print("=== STEP 1: Dataset Summary ===")
print(f"\nGender distribution in test set:")
print(df_test['sex'].value_counts())
print(f"\nPositive outcome rate by gender:")
gender_outcome = df_test.groupby('sex')['income'].agg(['sum', 'count', 'mean'])
gender_outcome.columns = ['positive_outcomes', 'total', 'positive_rate']
print(gender_outcome)

In [ ]:
# Step 2: Train a simple logistic regression classifier on the training set
# and compute fairness metrics on the test set

# Prepare training data
X_train = dataset_train.features
y_train = dataset_train.labels.flatten()

# Train a logistic regression classifier
clf = LogisticRegression(max_iter=1000, random_state=42)
clf.fit(X_train, y_train)

# Make predictions on test set
X_test = dataset_test.features
y_pred = clf.predict(X_test)

# Create a BinaryLabelDataset with predictions for AIF360
from aif360.datasets import BinaryLabelDataset

# Prepare the dataset with predictions
dataset_pred = dataset_test.copy()
dataset_pred.labels = y_pred.reshape(-1, 1)

# Compute fairness metrics using AIF360
metric = ClassificationMetric(dataset_test, dataset_pred, 
                              unprivileged_groups=[{adult_data.protected_attribute_names[0]: 0}],
                              privileged_groups=[{adult_data.protected_attribute_names[0]: 1}])

# Extract three key fairness metrics
dp_diff = metric.statistical_parity_difference()
eod_diff = metric.equalized_odds_difference()
di_ratio = metric.disparate_impact()

print("\n=== STEP 2: Fairness Metrics ===")
print(f"Demographic Parity Difference: {dp_diff:.4f}")
print(f"Equalized Odds Difference: {eod_diff:.4f}")
print(f"Disparate Impact Ratio: {di_ratio:.4f}")

## Expected Outputs

After running the cells above, you should see:

1. **Step 1 output**: A table showing gender distribution and positive-outcome rates by gender. You should see two gender groups (e.g., 0 and 1, representing different sexes) with their total counts and income positive rates.

2. **Step 2 output**: Three fairness metric values printed as floats with four decimal places:
   - Demographic Parity Difference (expected range: -1.0 to 1.0; 0 is perfect parity)
   - Equalized Odds Difference (expected range: -1.0 to 1.0)
   - Disparate Impact Ratio (expected range: 0.0 to 2.0+; 1.0 is perfect parity)

If metrics appear very close to 0, the classifier is highly fair on those dimensions. If they are large positive or negative, disparity is present.

## Questions and Reflection

Answer the following questions based on the metrics you computed:

1. Comparing the positive-outcome rates by gender from Step 1 and the Demographic Parity Difference from Step 2, which gender group had a higher prediction rate for positive income? Is the Demographic Parity Difference positive or negative, and what does the sign tell you?

2. The Disparate Impact ratio is sometimes reported as "four-fifths rule" in regulatory contexts (ratio >= 0.8 means no action required). Does your model meet the four-fifths rule for gender? If not, what is the implied impact on the underrepresented group?

In [ ]:
# Verification: Check that all three metrics were computed and are numeric
print("\n=== STEP 3 & 4: Bias Categorization and Audit Summary ===")

# Verify metrics exist and are numeric
assert isinstance(dp_diff, (int, float)) and np.isfinite(dp_diff), "Demographic Parity Difference should be a finite number"
assert isinstance(eod_diff, (int, float)) and np.isfinite(eod_diff), "Equalized Odds Difference should be a finite number"
assert isinstance(di_ratio, (int, float)) and np.isfinite(di_ratio), "Disparate Impact Ratio should be a finite number"

print("Verification: All three fairness metrics computed successfully.")
print(f"\nMetrics summary:")
print(f"  Demographic Parity Difference: {dp_diff:.4f}")
print(f"  Equalized Odds Difference: {eod_diff:.4f}")
print(f"  Disparate Impact Ratio: {di_ratio:.4f}")

# Check if model meets threshold
meets_threshold = abs(dp_diff) <= 0.1
print(f"\nModel meets Demographic Parity threshold (|DP_diff| <= 0.1): {meets_threshold}")

print("\n✓ Verification passed: Proceed to write your audit summary.")

## Optional Challenges

1. **Equal Opportunity Analysis**: Recompute the metrics filtering only to qualified applicants (ground truth = 1), and report TPR for each gender group separately. Does the model satisfy Equal Opportunity (equal TPR across genders)?

2. **Feature Inspection**: Use the trained classifier's feature importances (or coefficient magnitudes for logistic regression) to identify which features the model relies on most. Do any of these features correlate strongly with the protected attribute (gender)? This may suggest measurement bias.

3. **Re-training with Balanced Classes**: Retrain the classifier using class weights to balance the positive and negative outcome classes, and recompute the three fairness metrics. Do the metrics improve? Does accuracy on the test set change?